# Notebook 11 — Random Forest benchmark vs clasificador por umbrales

**Objetivo:** comparar formalmente el clasificador supervisado Random Forest con el clasificador por umbrales NDVI/CMRI sobre la misma imagen Sentinel-2 y el mismo AOI.

## Estrategia

GEE rechaza payloads > 10 MB y la cartografía INVEMAR pesa ~22 MB. Por eso:

1. **Muestreo + entrenamiento en GEE** usando **ESA WorldCover v200** como referencia base (asset nativo de GEE).
2. **Evaluación primaria** del RF contra WorldCover dentro de GEE.
3. **Evaluación secundaria** contra INVEMAR localmente: se exporta el raster RF como GeoTIFF y se compara con la cartografía INVEMAR rasterizada con `rasterio`.

In [ ]:
import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.features import rasterize
from shapely.validation import make_valid
from pathlib import Path

try:
    ee.Initialize(project='basic-buttress-338101')
except Exception:
    import google.auth
    creds, _ = google.auth.default()
    ee.Initialize(credentials=creds, project='basic-buttress-338101')

ROOT = Path('..').resolve()
OUT_TABLES = ROOT / 'outputs' / 'tables'
OUT_FIGURES = ROOT / 'outputs' / 'figures'
OUT_RASTERS = ROOT / 'outputs' / 'rasters'
OUT_RASTERS.mkdir(parents=True, exist_ok=True)

print(f'GEE OK · ROOT={ROOT}')

In [ ]:
# AOI acotado
AOI_PATH = ROOT / 'data' / 'raw' / 'cgsm_aoi_acotado_4326.geojson'
gdf_aoi = gpd.read_file(AOI_PATH).to_crs(4326)
aoi = ee.Geometry(gdf_aoi.geometry.union_all().__geo_interface__)
print('AOI cargado')

In [ ]:
# Composite Sentinel-2 mediana del periodo actual + auxiliares
def mask_s2(img):
    qa = img.select('QA60')
    return img.updateMask(qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0)))

def add_indices(img):
    ndvi = img.normalizedDifference(['B8','B4']).rename('NDVI')
    ndwi = img.normalizedDifference(['B3','B8']).rename('NDWI')
    cmri = ndvi.subtract(ndwi).rename('CMRI')
    return img.addBands([ndvi, ndwi, cmri])

bandas = ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12','NDVI','NDWI','CMRI']

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filterDate('2024-07-01', '2025-06-30')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .map(mask_s2).map(add_indices))

imagen = s2.median().select(bandas).clip(aoi)
srtm = ee.Image('USGS/SRTMGL1_003').clip(aoi).rename('elev')
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').clip(aoi)
dist_agua = (jrc.gt(30).fastDistanceTransform().sqrt().multiply(30).rename('dist_agua'))

stack = imagen.addBands([srtm, dist_agua])
bandas_completas = bandas + ['elev', 'dist_agua']
print(f'Stack: {len(bandas_completas)} bandas')

In [ ]:
# Referencia para muestreo: WorldCover v200 (nativo de GEE, sin upload)
worldcover = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map').clip(aoi)
wc_mangrove = worldcover.eq(95).rename('is_mangrove')
print('WorldCover v200 listo (clase 95 = manglar)')

In [ ]:
# Muestreo estratificado SOBRE WorldCover (todo dentro de GEE)
n_clase = 500

muestras = wc_mangrove.stratifiedSample(
    numPoints=n_clase,
    classBand='is_mangrove',
    region=aoi, scale=10, seed=42, geometries=True
)

muestras = stack.sampleRegions(
    collection=muestras, properties=['is_mangrove'],
    scale=10, geometries=True
)

n_total = muestras.size().getInfo()
print(f'Puntos muestreados: {n_total}')

In [ ]:
# K-fold (K=5)
K = 5
muestras_kf = muestras.randomColumn('fold', seed=42)

filas = []
for k in range(K):
    fmin, fmax = k/K, (k+1)/K
    test = muestras_kf.filter(ee.Filter.And(
        ee.Filter.gte('fold', fmin), ee.Filter.lt('fold', fmax)))
    train = muestras_kf.filter(ee.Filter.Or(
        ee.Filter.lt('fold', fmin), ee.Filter.gte('fold', fmax)))

    clf = (ee.Classifier.smileRandomForest(numberOfTrees=100, minLeafPopulation=5)
             .train(features=train, classProperty='is_mangrove',
                    inputProperties=bandas_completas))

    cm = test.classify(clf).errorMatrix('is_mangrove', 'classification').array().getInfo()
    tn, fp = cm[0]; fn, tp = cm[1]
    p = tp/(tp+fp) if (tp+fp) else 0
    r = tp/(tp+fn) if (tp+fn) else 0
    sp = tn/(tn+fp) if (tn+fp) else 0
    f1 = 2*p*r/(p+r) if (p+r) else 0
    oa = (tp+tn)/(tp+tn+fp+fn)
    filas.append({'fold': k+1, 'F1': f1, 'Precision': p, 'Recall': r, 'Specificity': sp, 'OA': oa})
    print(f'Fold {k+1}: F1={f1:.3f} P={p:.3f} R={r:.3f}')

df_kfold = pd.DataFrame(filas)
print(f'\nMedia K-fold: F1={df_kfold.F1.mean():.3f} ± {df_kfold.F1.std():.3f}')

In [ ]:
# RF final sobre todas las muestras → aplicar a la imagen
clf_final = (ee.Classifier.smileRandomForest(numberOfTrees=100, minLeafPopulation=5)
               .train(features=muestras_kf, classProperty='is_mangrove',
                      inputProperties=bandas_completas))

mapa_rf = stack.classify(clf_final).rename('rf_mangrove')
print('Mapa RF generado en GEE')

In [ ]:
# Evaluación RF vs WorldCover (todo en GEE, sin INVEMAR aún)
def metricas_pixel(pred, ref, region, escala=25):
    cm = pred.multiply(2).add(ref).rename('cm').reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=region, scale=escala, maxPixels=1e10
    ).getInfo()['cm']
    tn = int(cm.get('0', 0)); fn = int(cm.get('1', 0))
    fp = int(cm.get('2', 0)); tp = int(cm.get('3', 0))
    p = tp/(tp+fp) if (tp+fp) else 0
    r = tp/(tp+fn) if (tp+fn) else 0
    sp = tn/(tn+fp) if (tn+fp) else 0
    f1 = 2*p*r/(p+r) if (p+r) else 0
    oa = (tp+tn)/(tp+tn+fp+fn) if (tp+tn+fp+fn) else 0
    return {'F1': f1, 'Precision': p, 'Recall': r, 'Specificity': sp,
            'OverallAccuracy': oa, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn}

m_rf_wc = metricas_pixel(mapa_rf, wc_mangrove, aoi)
print('=== RF vs WorldCover ===')
for k, v in m_rf_wc.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

In [ ]:
# ============================================================
# INVEMAR simplificado para GEE (evita límite de 10 MB)
# ============================================================
INVEMAR_PATH = ROOT / 'data' / 'validation' / 'invemar_manglar_25k.geojson'
gdf_invemar = gpd.read_file(INVEMAR_PATH).to_crs(4326)
gdf_invemar = gpd.clip(gdf_invemar, gdf_aoi)
gdf_invemar['geometry'] = gdf_invemar.geometry.apply(make_valid)
print(f'INVEMAR clip al AOI: {len(gdf_invemar)} polígonos')

# Calcular tamaño antes de simplificar
pre_size = gdf_invemar.to_json()
print(f'Tamaño GeoJSON crudo: {len(pre_size)/1024/1024:.1f} MB')

# Reproyectar a CRS métrico, simplificar, volver a 4326
# Tolerancia 25 m = misma resolución de evaluación, sin perder detalle relevante
gdf_inv_proj = gdf_invemar.to_crs(9377)  # MAGNA-SIRGAS metros
gdf_inv_proj['geometry'] = gdf_inv_proj.geometry.simplify(
    tolerance=25, preserve_topology=True)
gdf_inv_simp = gdf_inv_proj.to_crs(4326)

# Filtrar geometrías vacías o nulas tras simplificación
gdf_inv_simp = gdf_inv_simp[gdf_inv_simp.geometry.notna()]
gdf_inv_simp = gdf_inv_simp[~gdf_inv_simp.geometry.is_empty]

post_size = gdf_inv_simp.to_json()
print(f'Tamaño GeoJSON simplificado: {len(post_size)/1024/1024:.2f} MB')

# Conversión a FeatureCollection (debe ser < 10 MB)
invemar_fc = geemap.gdf_to_ee(gdf_inv_simp)
print(f'INVEMAR subido a GEE: {invemar_fc.size().getInfo()} features')

In [ ]:
# ============================================================
# RF vs INVEMAR — evaluación por muestreo aleatorio (no pixel-a-pixel)
# ============================================================

# Crear raster INVEMAR de manera más eficiente con paint() (no reduceToImage)
invemar_raster = (ee.Image(0).byte()
                    .paint(invemar_fc, 1)
                    .clip(aoi)
                    .rename('is_mangrove'))

# Stack RF + INVEMAR
eval_stack = mapa_rf.rename('rf').addBands(invemar_raster.rename('inv'))

# Muestrear 10000 puntos aleatorios — estadísticamente equivalente al pixel-a-pixel
print('Muestreando 10.000 puntos para evaluación...')
muestra_eval = eval_stack.sample(
    region=aoi,
    scale=25,
    numPixels=10000,
    seed=99,
    geometries=False
)

# Obtener resultados como diccionario
datos = muestra_eval.aggregate_array('rf').getInfo()
ref = muestra_eval.aggregate_array('inv').getInfo()

import numpy as np
pred = np.array(datos)
ref = np.array(ref)

tp = int(((pred==1) & (ref==1)).sum())
fp = int(((pred==1) & (ref==0)).sum())
fn = int(((pred==0) & (ref==1)).sum())
tn = int(((pred==0) & (ref==0)).sum())

p = tp/(tp+fp) if (tp+fp) else 0
r = tp/(tp+fn) if (tp+fn) else 0
sp = tn/(tn+fp) if (tn+fp) else 0
f1 = 2*p*r/(p+r) if (p+r) else 0
oa = (tp+tn)/(tp+tn+fp+fn) if (tp+tn+fp+fn) else 0

m_rf_inv = {'F1': f1, 'Precision': p, 'Recall': r, 'Specificity': sp,
            'OverallAccuracy': oa, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
            'n_samples': int(tp+fp+fn+tn)}

print('=== RF vs INVEMAR (muestreo n=10000) ===')
for k, v in m_rf_inv.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

In [ ]:
# Tabla comparativa final con ambas referencias
umbrales = {
    'INVEMAR': {'F1':0.583,'Precision':0.811,'Recall':0.454,'Specificity':0.954,'OverallAccuracy':0.803},
    'WorldCover': {'F1':0.548,'Precision':0.768,'Recall':0.426,'Specificity':0.944,'OverallAccuracy':0.788},
}

tabla = pd.DataFrame([
    {'Método':'Umbrales NDVI/CMRI', 'Referencia':'INVEMAR 1:25.000', **umbrales['INVEMAR']},
    {'Método':'Umbrales NDVI/CMRI', 'Referencia':'ESA WorldCover v200', **umbrales['WorldCover']},
    {'Método':'Random Forest', 'Referencia':'INVEMAR 1:25.000',
     'F1':m_rf_inv['F1'], 'Precision':m_rf_inv['Precision'],
     'Recall':m_rf_inv['Recall'], 'Specificity':m_rf_inv['Specificity'],
     'OverallAccuracy':m_rf_inv['OverallAccuracy']},
    {'Método':'Random Forest', 'Referencia':'ESA WorldCover v200',
     'F1':m_rf_wc['F1'], 'Precision':m_rf_wc['Precision'],
     'Recall':m_rf_wc['Recall'], 'Specificity':m_rf_wc['Specificity'],
     'OverallAccuracy':m_rf_wc['OverallAccuracy']},
]).round(3)

print(tabla.to_string(index=False))

tabla.to_csv(OUT_TABLES / 'benchmark_rf_vs_umbrales.csv', index=False)
print(f'\n✓ outputs/tables/benchmark_rf_vs_umbrales.csv')

In [ ]:
# Importancia de variables
exp = clf_final.explain().getInfo()
imp = exp.get('importance', {})
df_imp = pd.DataFrame(sorted(imp.items(), key=lambda x: -x[1]),
                       columns=['variable', 'importance'])
print(df_imp)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(df_imp['variable'][::-1], df_imp['importance'][::-1], color='#1f5a4b')
ax.set_xlabel('Importancia (Gini)')
ax.set_title('Random Forest · Importancia de variables predictoras')
plt.tight_layout()
plt.savefig(OUT_FIGURES / 'rf_feature_importance.png', dpi=180, bbox_inches='tight')
plt.show()

df_imp.to_csv(OUT_TABLES / 'rf_feature_importance.csv', index=False)
print('✓ outputs/figures/rf_feature_importance.png')